In [30]:
# Generic inputs for most ML tasks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
# This is new
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

pd.options.display.float_format = '{:,.2f}'.format

# setup interactive notebook mode
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython.display import display, HTML

In [31]:
# fetch data 

mco_syr_sw_data = pd.read_csv('flight_data\mco_syr_sw_22_23.csv')
mco_syr_jb_data = pd.read_csv('flight_data\mco_syr_jb_22_23.csv')
jfk_syr_jb_data = pd.read_csv('flight_data\jfk_syr_jb_22_23.csv')
jfk_syr_end_data = pd.read_csv('flight_data\jfk_syr_end_22_23.csv')
ord_syr_ua_data = pd.read_csv('flight_data\ord_syr_ua_22_23.csv')
ord_syr_aa_data = pd.read_csv('flight_data\ord_syr_aa_22_23.csv')
mco_syr_jb_data['dep_order'] = 'early'
ord_syr_ua_data['dep_order'] = 'early'
jfk_syr_jb_data['dep_order'] = 'early'
mco_syr_sw_data['dep_order'] = 'latter'
jfk_syr_end_data['dep_order'] = 'latter'
ord_syr_aa_data['dep_order'] = 'latter'
#['dep_order'] = 'early'
# mco_syr_sw_data.head()
# mco_syr_jb_data.head()
# jfk_syr_jb_data.head()
# jfk_syr_end_data.head()
# ord_syr_ua_data.head()
# ord_syr_aa_data.head()

In [32]:
dfs = [ord_syr_aa_data,ord_syr_ua_data,jfk_syr_end_data,jfk_syr_jb_data,mco_syr_jb_data,mco_syr_sw_data]
main_data = pd.concat(dfs,axis = 0)
main_data.head()
len(main_data)

,Unnamed: 0,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),...,Wheels-on Time,Taxi-In time (Minutes),arr_Delay_Carrier,arr_Delay_Weather,arr_Delay_National_Aviation_System,arr_Delay_Security,arr_Delay_Late_Aircraft_Arrival,arr_hour,arr_day,arr_year
0,0,MQ,2022-01-01,4316,N900AE,SYR,15:28,0:00,107,0,...,0:00,0,0,0,0,0,0,18,5,2022
1,1,MQ,2023-01-01,3392,N283NN,SYR,18:25,18:24,111,104,...,21:02,6,0,0,0,0,0,21,6,2023
2,2,MQ,2023-01-01,3518,N213NN,SYR,13:05,13:01,117,96,...,15:33,4,0,0,0,0,0,16,6,2023
3,3,MQ,2022-01-02,4134,N900AE,SYR,10:25,10:21,111,146,...,13:43,4,0,0,31,0,0,13,6,2022
4,4,MQ,2022-01-02,4253,N238NN,SYR,17:25,17:51,107,116,...,20:40,7,0,26,9,0,0,20,6,2022


5623

In [33]:
main_data.dtypes



Unnamed: 0                             int64
Carrier_Code                          object
Date                                  object
Flight_Number                          int64
Tail_Number                           object
Destination_Airport                   object
Scheduled departure time              object
Actual departure time                 object
Scheduled elapsed time (Minutes)       int64
Actual elapsed time (Minutes)          int64
Departure delay (Minutes)              int64
Wheels-off time                       object
Taxi-Out time (Minutes)                int64
dep_Delay_Carrier                      int64
dep_Delay_Weather                      int64
dep_Delay_National_Aviation_System     int64
dep_Delay_Security                     int64
dep_Delay_Late_Aircraft_Arrival        int64
dep_hour                               int64
dep_day                                int64
dep_year                               int64
dep_order                             object
Origin_Air

In [34]:
sub_data = main_data.drop(columns = ['Unnamed: 0','Destination_Airport','Actual departure time','Scheduled elapsed time (Minutes)','Actual elapsed time (Minutes)',
                         'Departure delay (Minutes)','Wheels-off time','Taxi-Out time (Minutes)','dep_Delay_Carrier',
                         'dep_Delay_Weather','dep_Delay_National_Aviation_System','dep_Delay_Security','dep_Delay_Late_Aircraft_Arrival','Actual Arrival Time','Wheels-on Time','Taxi-In time (Minutes)','arr_Delay_Carrier','arr_Delay_Weather',
                         'arr_Delay_National_Aviation_System','arr_Delay_Security','arr_Delay_Late_Aircraft_Arrival','arr_year'])

In [35]:
sub_data.dtypes


Carrier_Code                object
Date                        object
Flight_Number                int64
Tail_Number                 object
Scheduled departure time    object
dep_hour                     int64
dep_day                      int64
dep_year                     int64
dep_order                   object
Origin_Airport              object
Scheduled Arrival Time      object
Arrival Delay (Minutes)      int64
arr_hour                     int64
arr_day                      int64
dtype: object

In [36]:
sub_data.head()

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_year,dep_order,Origin_Airport,Scheduled Arrival Time,Arrival Delay (Minutes),arr_hour,arr_day
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,2022,latter,ORD,18:15,0,18,5
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,2023,latter,ORD,21:16,-8,21,6
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,2023,latter,ORD,16:02,-25,16,6
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,2022,latter,ORD,13:16,31,13,6
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,2022,latter,ORD,20:12,35,20,6


In [37]:
sub_data['dep_min'] = sub_data['Scheduled departure time'].str.split(":").str[1].astype('object')
sub_data['arr_min'] = main_data['Scheduled Arrival Time'].str.split(":").str[1].astype('object')
sub_data['Flight_Number'] = main_data['Flight_Number'].astype('object')
sub_data['dep_hour'] = main_data['dep_hour'].astype('object')
sub_data['dep_day'] = main_data['dep_day'].astype('object')
sub_data['arr_hour'] = main_data['arr_hour'].astype('object')
sub_data['arr_day'] = main_data['arr_day'].astype('object')
conditions = [
    (sub_data['Arrival Delay (Minutes)'] > 5),
    (sub_data['Arrival Delay (Minutes)'] >=-5) & (sub_data['Arrival Delay (Minutes)'] <= 5),
    (sub_data['Arrival Delay (Minutes)'] < -5)
]
choices = [2,1,0]
sub_data['arr_status'] = np.select(conditions, choices)
sub_data.dtypes
sub_data.head()

Carrier_Code                object
Date                        object
Flight_Number               object
Tail_Number                 object
Scheduled departure time    object
dep_hour                    object
dep_day                     object
dep_year                     int64
dep_order                   object
Origin_Airport              object
Scheduled Arrival Time      object
Arrival Delay (Minutes)      int64
arr_hour                    object
arr_day                     object
dep_min                     object
arr_min                     object
arr_status                   int32
dtype: object

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_year,dep_order,Origin_Airport,Scheduled Arrival Time,Arrival Delay (Minutes),arr_hour,arr_day,dep_min,arr_min,arr_status
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,2022,latter,ORD,18:15,0,18,5,28,15,1
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,2023,latter,ORD,21:16,-8,21,6,25,16,0
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,2023,latter,ORD,16:02,-25,16,6,05,02,0
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,2022,latter,ORD,13:16,31,13,6,25,16,2
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,2022,latter,ORD,20:12,35,20,6,25,12,2


In [38]:
latter_data = sub_data[sub_data['dep_order'] == 'latter']
len(latter_data)

2927

In [39]:
early_data = sub_data[sub_data['dep_order'] == 'early']
len(early_data)

2696

In [40]:
# latter_data['dep_year'].value_counts()

In [41]:
# early_data['dep_year'].value_counts()

In [42]:
# latter_data['Carrier_Code'].value_counts()

In [25]:
# early_data['Carrier_Code'].value_counts()

In [43]:
merged_df = pd.merge(latter_data, early_data, on=['Date', 'Origin_Airport'], how = 'left')
merged_df.head()
len(merged_df)

,Carrier_Code_x,Date,Flight_Number_x,Tail_Number_x,Scheduled departure time_x,dep_hour_x,dep_day_x,dep_year_x,dep_order_x,Origin_Airport,...,dep_day_y,dep_year_y,dep_order_y,Scheduled Arrival Time_y,Arrival Delay (Minutes)_y,arr_hour_y,arr_day_y,dep_min_y,arr_min_y,arr_status_y
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,2022,latter,ORD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,2023,latter,ORD,...,6,"2,023.00",early,23:57,-10.00,23,6,10,57,0.00
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,2023,latter,ORD,...,6,"2,023.00",early,23:57,-10.00,23,6,10,57,0.00
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,2022,latter,ORD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,2022,latter,ORD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


4568

In [44]:
merged_df.head(20)

,Carrier_Code_x,Date,Flight_Number_x,Tail_Number_x,Scheduled departure time_x,dep_hour_x,dep_day_x,dep_year_x,dep_order_x,Origin_Airport,...,dep_day_y,dep_year_y,dep_order_y,Scheduled Arrival Time_y,Arrival Delay (Minutes)_y,arr_hour_y,arr_day_y,dep_min_y,arr_min_y,arr_status_y
0,MQ,2022-01-01,4316,N900AE,15:28,15,5,2022,latter,ORD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MQ,2023-01-01,3392,N283NN,18:25,18,6,2023,latter,ORD,...,6,"2,023.00",early,23:57,-10.00,23,6,10,57,0.00
2,MQ,2023-01-01,3518,N213NN,13:05,13,6,2023,latter,ORD,...,6,"2,023.00",early,23:57,-10.00,23,6,10,57,0.00
3,MQ,2022-01-02,4134,N900AE,10:25,10,6,2022,latter,ORD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MQ,2022-01-02,4253,N238NN,17:25,17,6,2022,latter,ORD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,MQ,2022-01-02,4316,N922AE,15:28,15,6,2022,latter,ORD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,MQ,2023-01-02,3392,N248NN,17:33,17,0,2023,latter,ORD,...,0,"2,023.00",early,21:07,-21.00,21,0,14,07,0.00
7,MQ,2023-01-02,3518,N263NN,13:05,13,0,2023,latter,ORD,...,0,"2,023.00",early,21:07,-21.00,21,0,14,07,0.00
8,MQ,2022-01-03,4134,N606AE,10:25,10,0,2022,latter,ORD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,MQ,2022-01-03,4253,N200NN,17:25,17,0,2022,latter,ORD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
merged_df.columns

Index(['Carrier_Code_x', 'Date', 'Flight_Number_x', 'Tail_Number_x',
       'Scheduled departure time_x', 'dep_hour_x', 'dep_day_x', 'dep_year_x',
       'dep_order_x', 'Origin_Airport', 'Scheduled Arrival Time_x',
       'Arrival Delay (Minutes)_x', 'arr_hour_x', 'arr_day_x', 'dep_min_x',
       'arr_min_x', 'arr_status_x', 'Carrier_Code_y', 'Flight_Number_y',
       'Tail_Number_y', 'Scheduled departure time_y', 'dep_hour_y',
       'dep_day_y', 'dep_year_y', 'dep_order_y', 'Scheduled Arrival Time_y',
       'Arrival Delay (Minutes)_y', 'arr_hour_y', 'arr_day_y', 'dep_min_y',
       'arr_min_y', 'arr_status_y'],
      dtype='object')

In [46]:
merged_df['hour_diff']= merged_df['dep_hour_x'] - merged_df['dep_hour_y']
merged_df1 = merged_df[merged_df['hour_diff'] ==0 ]
merged_df1 = merged_df1[merged_df1['dep_min_x'] > merged_df1['dep_min_y']]
merged_df2 = merged_df[merged_df['hour_diff'] ==1]
merged_df = pd.concat([merged_df1,merged_df2])
len(merged_df)

675

In [47]:
merged_df['dep_year_x'].value_counts()

dep_year_x
2023    500
2022    175
Name: count, dtype: int64

In [49]:
merged_df.to_csv('flight_data\\all_lat_ear.csv')

In [50]:
merged_df.drop(columns= ['Carrier_Code_x', 'Date', 'Flight_Number_x', 'Tail_Number_x',
       'Scheduled departure time_x', 'dep_year_x',
       'dep_order_x', 'Scheduled Arrival Time_x',
       'Arrival Delay (Minutes)_x', 'Carrier_Code_y', 'Flight_Number_y',
       'Tail_Number_y', 'Scheduled departure time_y', 'dep_hour_y',
       'dep_day_y', 'dep_year_y', 'dep_order_y', 'Scheduled Arrival Time_y',
       'Arrival Delay (Minutes)_y', 'arr_hour_y', 'arr_day_y', 'dep_min_y',
       'arr_min_y','hour_diff'],inplace = True)

merged_df.dtypes

dep_hour_x         object
dep_day_x          object
Origin_Airport     object
arr_hour_x         object
arr_day_x          object
dep_min_x          object
arr_min_x          object
arr_status_x        int32
arr_status_y      float64
dtype: object

In [51]:
merged_df['arr_status_y'] = merged_df['arr_status_y'].astype('int64').astype('object')
merged_df.head()
merged_df.dtypes
su_data = merged_df

,dep_hour_x,dep_day_x,Origin_Airport,arr_hour_x,arr_day_x,dep_min_x,arr_min_x,arr_status_x,arr_status_y
313,18,1,ORD,21,1,30,25,1,2
314,18,2,ORD,21,2,36,32,1,2
317,18,3,ORD,21,3,36,32,0,0
328,18,0,ORD,21,0,36,32,2,2
331,18,1,ORD,21,1,36,32,2,1


dep_hour_x        object
dep_day_x         object
Origin_Airport    object
arr_hour_x        object
arr_day_x         object
dep_min_x         object
arr_min_x         object
arr_status_x       int32
arr_status_y      object
dtype: object

In [52]:
# su_data = sub_data.drop(columns = ['dep_order'])
su_data.head()

,dep_hour_x,dep_day_x,Origin_Airport,arr_hour_x,arr_day_x,dep_min_x,arr_min_x,arr_status_x,arr_status_y
313,18,1,ORD,21,1,30,25,1,2
314,18,2,ORD,21,2,36,32,1,2
317,18,3,ORD,21,3,36,32,0,0
328,18,0,ORD,21,0,36,32,2,2
331,18,1,ORD,21,1,36,32,2,1


In [53]:
su_data = pd.get_dummies(su_data, drop_first = True)
#su_data['arr_status'] = su_data['arr_status'].astype('object')
su_data.head()
su_data.dtypes

,arr_status_x,dep_hour_x_12,dep_hour_x_13,dep_hour_x_14,dep_hour_x_15,dep_hour_x_18,dep_hour_x_19,dep_hour_x_22,dep_hour_x_23,dep_day_x_1,...,arr_min_x_49,arr_min_x_53,arr_min_x_54,arr_min_x_55,arr_min_x_56,arr_min_x_57,arr_min_x_58,arr_min_x_59,arr_status_y_1,arr_status_y_2
313,1,False,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True
314,1,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
317,0,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
328,2,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
331,2,False,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,True,False


arr_status_x      int32
dep_hour_x_12      bool
dep_hour_x_13      bool
dep_hour_x_14      bool
dep_hour_x_15      bool
                  ...  
arr_min_x_57       bool
arr_min_x_58       bool
arr_min_x_59       bool
arr_status_y_1     bool
arr_status_y_2     bool
Length: 87, dtype: object

In [54]:
X_train, X_test, y_train, y_test = train_test_split(su_data.drop(columns = ['arr_status_x']), su_data['arr_status_x'], test_size=0.2, stratify = su_data['arr_status_x'], random_state=50)
# In the above split the stratify = y essentially makes sure the fractions of the classification is maintained
X_train
X_test
y_train.dtypes
y_test

,dep_hour_x_12,dep_hour_x_13,dep_hour_x_14,dep_hour_x_15,dep_hour_x_18,dep_hour_x_19,dep_hour_x_22,dep_hour_x_23,dep_day_x_1,dep_day_x_2,...,arr_min_x_49,arr_min_x_53,arr_min_x_54,arr_min_x_55,arr_min_x_56,arr_min_x_57,arr_min_x_58,arr_min_x_59,arr_status_y_1,arr_status_y_2
1650,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,True,False,False,False,True
1279,False,False,False,False,False,True,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
2197,False,False,False,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
4250,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
142,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3991,False,False,False,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,True,False,True
276,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
971,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3855,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


,dep_hour_x_12,dep_hour_x_13,dep_hour_x_14,dep_hour_x_15,dep_hour_x_18,dep_hour_x_19,dep_hour_x_22,dep_hour_x_23,dep_day_x_1,dep_day_x_2,...,arr_min_x_49,arr_min_x_53,arr_min_x_54,arr_min_x_55,arr_min_x_56,arr_min_x_57,arr_min_x_58,arr_min_x_59,arr_status_y_1,arr_status_y_2
2285,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3923,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4401,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3574,False,False,False,True,False,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
4239,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3125,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,True,False
2179,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
54,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3209,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,True


dtype('int32')

2285    1
3923    0
4401    0
3574    0
4239    0
       ..
3125    0
2179    2
54      2
3209    2
2355    0
Name: arr_status_x, Length: 135, dtype: int32

In [55]:
#model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'ovr', penalty = 'none', max_iter = 1000)
model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'multinomial', penalty = 'none', max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 10000)

# While using multiclass case do multi_class = 'ovr' or 'auto'; can also try other solvers
# While doing regularization, use penalty = 'l2' and also C = 10.0 (need to try other values too)

model.fit(X_train, y_train) 

# The following gives the mean accuracy on the given data and labels
model.score(X_train, y_train) 

# This is the coefficient Beta_1, ..., Beta_7
model.coef_

# This is the coefficient Beta_0
model.intercept_

C:\Users\gurud\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1182: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(


LogisticRegression(max_iter=1000, multi_class='multinomial', penalty='none')

0.6185185185185185

array([[ 2.13625439e+00,  2.67239381e-01, -6.84519456e-01,
         4.74017254e+00, -4.12234238e+00,  3.94711810e+00,
        -1.70282630e+00, -1.17418214e+01,  3.02657303e-01,
         9.06263740e-02,  7.62305613e-02,  1.33417847e-01,
         1.26114468e-01,  1.48297054e-01,  2.13625439e+00,
        -7.30444582e+00,  1.14081226e+01,  5.64703911e-01,
         2.13625439e+00,  2.67239381e-01, -1.04780468e+00,
         5.10345776e+00,  1.78749632e+00, -1.96272059e+00,
         4.73017060e+00,  3.02657303e-01,  9.06263740e-02,
         7.62305613e-02,  1.33417847e-01,  1.26114468e-01,
         1.48297054e-01, -1.76517130e+01, -1.31074458e-01,
        -2.63588682e+00,  1.27682403e-01, -4.91751405e+00,
         0.00000000e+00,  2.23789964e+00,  1.71505856e+00,
         0.00000000e+00, -1.30653876e+01,  3.99197611e+00,
         1.02718061e-01, -1.19922347e+00,  1.83861981e+00,
         7.52163162e-02,  0.00000000e+00, -1.35684998e+00,
        -2.55871828e-01,  7.47055126e+00,  1.74183811e+0

array([ 4.81210143, -1.32568738, -3.48641405])

In [56]:
model.score(X_test,y_test)

0.5259259259259259

In [57]:

from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=50, min_samples_split = 12, min_samples_leaf = 6, max_depth = 4, n_estimators = 300)

gb = gb.fit(X_train, y_train) 
gb.score(X_train, y_train) 

0.7444444444444445

In [58]:
gb.score(X_test,y_test)

0.42962962962962964